## 国土数値情報 駅別乗降客数データ (S12-25)

In [2]:
import zipfile
from pathlib import Path

import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)

# このノートブックは script/ に置かれている想定。リポジトリ直下を基準にする。
BASE_DIR = Path.cwd().parent if Path.cwd().name == "script" else Path.cwd()
ZIP_PATH = BASE_DIR / "data" / "S12-25_GML.zip"
assert ZIP_PATH.exists(), f"ZIP が見つかりません: {ZIP_PATH}"

In [3]:
# ZIP 内のファイル一覧を確認
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        print(name)

S12-25_GML/
S12-25_GML/KS-META-S12-25.xml
S12-25_GML/KsjAppSchema-S12-v3_3.xsd
S12-25_GML/Shift-JIS/
S12-25_GML/Shift-JIS/S12-25.xml
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.dbf
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.geojson
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.prj
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shp
S12-25_GML/Shift-JIS/S12-25_NumberOfPassengers.shx
S12-25_GML/UTF-8/
S12-25_GML/UTF-8/S12-25.xml
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.dbf
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.prj
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shp
S12-25_GML/UTF-8/S12-25_NumberOfPassengers.shx


In [4]:
# ZIP を解凍せず、geopandas の zip:// 仮想パスで直接 GeoJSON を読み込む
GEOJSON_INNER = "S12-25_GML/UTF-8/S12-25_NumberOfPassengers.geojson"
read_path = f"zip://{ZIP_PATH}!{GEOJSON_INNER}"

gdf = gpd.read_file(read_path)
print("行数 x 列数:", gdf.shape)
print("CRS:", gdf.crs)

行数 x 列数: (10534, 64)
CRS: EPSG:6668


In [5]:
# 属性列を S12 仕様の日本語名にリネーム
# 固定項目
rename_map = {
    "S12_001": "駅名",
    "S12_001c": "駅コード",
    "S12_001g": "グループコード",
    "S12_002": "運営会社",
    "S12_003": "路線名",
    "S12_004": "鉄道区分",
    "S12_005": "事業者種別",
}

# 年度ごとに [重複コード, データ有無コード, 備考, 乗降客数] が4列ずつ繰り返す
# 2011年度=S12_006〜S12_009 ... 2024年度=S12_058〜S12_061
for i, year in enumerate(range(2011, 2025)):
    base = 6 + i * 4
    rename_map[f"S12_{base:03d}"] = f"重複コード{year}"
    rename_map[f"S12_{base + 1:03d}"] = f"データ有無コード{year}"
    rename_map[f"S12_{base + 2:03d}"] = f"備考{year}"
    rename_map[f"S12_{base + 3:03d}"] = f"乗降客数{year}"

gdf = gdf.rename(columns=rename_map)
list(gdf.columns)

['駅名',
 '駅コード',
 'グループコード',
 '運営会社',
 '路線名',
 '鉄道区分',
 '事業者種別',
 '重複コード2011',
 'データ有無コード2011',
 '備考2011',
 '乗降客数2011',
 '重複コード2012',
 'データ有無コード2012',
 '備考2012',
 '乗降客数2012',
 '重複コード2013',
 'データ有無コード2013',
 '備考2013',
 '乗降客数2013',
 '重複コード2014',
 'データ有無コード2014',
 '備考2014',
 '乗降客数2014',
 '重複コード2015',
 'データ有無コード2015',
 '備考2015',
 '乗降客数2015',
 '重複コード2016',
 'データ有無コード2016',
 '備考2016',
 '乗降客数2016',
 '重複コード2017',
 'データ有無コード2017',
 '備考2017',
 '乗降客数2017',
 '重複コード2018',
 'データ有無コード2018',
 '備考2018',
 '乗降客数2018',
 '重複コード2019',
 'データ有無コード2019',
 '備考2019',
 '乗降客数2019',
 '重複コード2020',
 'データ有無コード2020',
 '備考2020',
 '乗降客数2020',
 '重複コード2021',
 'データ有無コード2021',
 '備考2021',
 '乗降客数2021',
 '重複コード2022',
 'データ有無コード2022',
 '備考2022',
 '乗降客数2022',
 '重複コード2023',
 'データ有無コード2023',
 '備考2023',
 '乗降客数2023',
 '重複コード2024',
 'データ有無コード2024',
 '備考2024',
 '乗降客数2024',
 'geometry']

In [6]:
gdf.info()
gdf.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 64 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   駅名            10534 non-null  object  
 1   駅コード          10297 non-null  object  
 2   グループコード       10297 non-null  object  
 3   運営会社          10534 non-null  object  
 4   路線名           10534 non-null  object  
 5   鉄道区分          10534 non-null  int32   
 6   事業者種別         10534 non-null  int32   
 7   重複コード2011     10534 non-null  int32   
 8   データ有無コード2011  10534 non-null  int32   
 9   備考2011        122 non-null    object  
 10  乗降客数2011      10534 non-null  int32   
 11  重複コード2012     10534 non-null  int32   
 12  データ有無コード2012  10534 non-null  int32   
 13  備考2012        144 non-null    object  
 14  乗降客数2012      10534 non-null  int32   
 15  重複コード2013     10534 non-null  int32   
 16  データ有無コード2013  10534 non-null  int32   
 17  備考2013        159 non-null    object  
 18

,駅名,駅コード,グループコード,運営会社,路線名,鉄道区分,事業者種別,重複コード2011,データ有無コード2011,備考2011,乗降客数2011,重複コード2012,データ有無コード2012,備考2012,乗降客数2012,重複コード2013,データ有無コード2013,備考2013,乗降客数2013,重複コード2014,データ有無コード2014,備考2014,乗降客数2014,重複コード2015,データ有無コード2015,備考2015,乗降客数2015,重複コード2016,データ有無コード2016,備考2016,乗降客数2016,重複コード2017,データ有無コード2017,備考2017,乗降客数2017,重複コード2018,データ有無コード2018,備考2018,乗降客数2018,重複コード2019,データ有無コード2019,備考2019,乗降客数2019,重複コード2020,データ有無コード2020,備考2020,乗降客数2020,重複コード2021,データ有無コード2021,備考2021,乗降客数2021,重複コード2022,データ有無コード2022,備考2022,乗降客数2022,重複コード2023,データ有無コード2023,備考2023,乗降客数2023,重複コード2024,データ有無コード2024,備考2024,乗降客数2024,geometry
0,二月田,010112,010112,九州旅客鉄道,指宿枕崎線,11,2,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,3,None,0,1,1,None,658,1,1,None,690,1,1,None,318,1,1,None,305,1,1,None,622,1,1,None,664,1,1,None,726,"LINESTRING (130.63035 31.25405, 130.62985 31.2..."
1,古島,010127,010127,沖縄都市モノレール,沖縄都市モノレール線,23,5,1,1,None,3907,1,1,None,3980,1,1,None,4572,1,1,None,4187,1,1,None,4648,1,1,None,4528,1,1,None,4819,1,1,None,5086,1,1,None,5143,1,1,None,3404,1,1,None,3534,1,1,None,4401,1,1,None,4903,1,1,None,5490,"LINESTRING (127.70279 26.23035, 127.70309 26.2..."
2,お台場海浜公園,004091,004091,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,14612,1,1,None,16130,1,1,None,16357,1,1,None,16515,1,1,None,17537,1,1,None,17488,1,1,None,16944,1,1,None,17165,1,1,None,16899,1,1,None,7703,1,1,None,8535,1,1,None,11171,1,1,None,13195,1,1,None,13435,"LINESTRING (139.77818 35.62961, 139.77888 35.63)"
3,東京国際クルーズターミナル,004128,004128,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,3767,1,1,None,3235,1,1,None,3407,1,1,None,3782,1,1,None,3682,1,1,None,3682,1,1,None,3590,1,1,None,3551,1,1,None,3191,1,1,None,1383,1,1,None,1984,1,1,None,2300,1,1,None,2963,1,1,None,3503,"LINESTRING (139.77333 35.62109, 139.77288 35.6..."
4,テレコムセンター,004144,004144,ゆりかもめ,東京臨海新交通臨海線,24,5,1,1,None,12112,1,1,None,12775,1,1,None,13366,1,1,None,14138,1,1,None,14069,1,1,None,14069,1,1,None,13744,1,1,None,13475,1,1,None,12140,1,1,None,6536,1,1,None,7631,1,1,None,8118,1,1,None,8505,1,1,None,9267,"LINESTRING (139.78001 35.61791, 139.77932 35.6..."
